In [18]:
# Imports
from pathlib import Path
from experiment.utils import TrainedModel, TrainedModelID, get_losses

import pandas as pd
import torch
from neuralhydrology.nh_run import start_run, eval_run, finetune
from experiment.eval import evaluate_models
import os
import yaml
import matplotlib.pyplot as plt

In [19]:
model = TrainedModel(TrainedModelID.SOTA_20)

df = pd.read_csv(model.metrics_file, dtype={'basin':str})
cutoff = 0.4
basin_data = df.loc[df['NSE'] < cutoff].sample(n=1)
basin = basin_data.basin.iloc[0]
nse = basin_data.NSE.iloc[0]
basin

'09484600'

In [55]:
# Add the path to the pre-trained model to the finetune config
file_path = "finetune.yml"

with open(file_path, "a") as fp:
    fp.write(f"\nbase_run_dir: {model.run_dir.absolute()}")

# Load the existing YAML data
with open(file_path, 'r') as f:
    data = yaml.safe_load(f)

data['experiment_name'] = f'basin_{basin}'  # Example modification

# Write back to the YAML file
with open(file_path, 'w') as f:
    yaml.dump(data, f)   


In [56]:
# Create a basin file with the basin we selected above
with open("finetune_basin.txt", "w") as fp:
    fp.write(basin)

In [57]:
# finetune
finetune(Path("finetune.yml"))

2024-09-18 21:10:45,069: Logging to /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/output.log initialized.
2024-09-18 21:10:45,070: ### Folder structure created at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600
2024-09-18 21:10:45,070: ### Start finetuning with pretrained model stored in /home/admin/Fine-Flood-Forecasts/experiment/models/runs/sota_20
2024-09-18 21:10:45,071: ### Run configurations for basin_09484600
2024-09-18 21:10:45,071: additional_feature_files: None
2024-09-18 21:10:45,072: batch_size: 256
2024-09-18 21:10:45,072: checkpoint_path: None
2024-09-18 21:10:45,072: clip_gradient_norm: 1
2024-09-18 21:10:45,073: clip_targets_to_zero: ['QObs(mm/d)']
2024-09-18 21:10:45,073: commit_hash: 6dde7b4
2024-09-18 21:10:45,074: data_dir: /home/admin/Fine-Flood-Forecasts/data/CAMELS_US
2024-09-18 21:10:45,074: dataset: camels_us
2024-09-18 21:10:45,074: device: cuda:0
2024-09-18 21:10:45,075: dynamic_inp

/home/admin/Fine-Flood-Forecasts/neuralhydrology/training/basetrainer.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(str(checkpo

# Epoch 1: 100%|██████████| 13/13 [00:00<00:00, 21.24it/s, Loss: 0.0078]
2024-09-18 21:10:45,929: Epoch 1 average loss: avg_loss: 0.01099, avg_total_loss: 0.01099
# Epoch 2: 100%|██████████| 13/13 [00:00<00:00, 21.23it/s, Loss: 0.0043]
2024-09-18 21:10:46,551: Epoch 2 average loss: avg_loss: 0.00857, avg_total_loss: 0.00857
# Epoch 3: 100%|██████████| 13/13 [00:00<00:00, 20.85it/s, Loss: 0.0046]
2024-09-18 21:10:47,184: Epoch 3 average loss: avg_loss: 0.00833, avg_total_loss: 0.00833
# Validation: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]
2024-09-18 21:10:48,090: Epoch 3 average validation loss: nan -- Median validation metrics: avg_loss: nan
# Epoch 4: 100%|██████████| 13/13 [00:00<00:00, 18.27it/s, Loss: 0.0106]
2024-09-18 21:10:48,804: Epoch 4 average loss: avg_loss: 0.00856, avg_total_loss: 0.00856
2024-09-18 21:10:48,811: Setting learning rate to 0.0005
# Epoch 5: 100%|██████████| 13/13 [00:00<00:00, 20.86it/s, Loss: 0.0050]
2024-09-18 21:10:49,438: Epoch 5 average loss: avg_l

In [58]:
run_dir = Path(os.path.abspath('')) / 'runs' / f'basin_{basin}'
config_file_path = run_dir / 'config.yml'
output_log = run_dir / 'output.log'

# plot the training and validation loss
train, val = get_losses(output_log)
train_epoch, train_ls = zip(*train.items()) 
val_epoch, val_ls = zip(*val.items()) 

plt.plot(train_epoch, train_ls)
plt.plot(val_epoch, val_ls)


ValueError: not enough values to unpack (expected 2, got 0)

In [59]:

finetuned_model = TrainedModel(config_file_path_or_experiment_name=config_file_path)
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='train')
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='test')

2024-09-18 21:11:01,870: Using the model weights from /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/model_epoch010.pt


/home/admin/Fine-Flood-Forecasts/neuralhydrology/evaluation/tester.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_file, m

# Evaluation: 100%|██████████| 1/1 [00:00<00:00,  3.71it/s]
2024-09-18 21:11:02,145: Stored metrics at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/train/model_epoch010/train_metrics.csv
2024-09-18 21:11:02,146: Stored results at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/train/model_epoch010/train_results.p


,Metric,sota_20,basin_09484600
0,NSE (mean),0.46,0.57
1,KGE (mean),0.45,0.6
2,MSE (mean),0.00,0.0
3,NSE (median),0.46,0.57
4,KGE (median),0.45,0.6
5,MSE (median),0.00,0.0


2024-09-18 21:11:02,173: Using the model weights from /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/model_epoch010.pt


/home/admin/Fine-Flood-Forecasts/neuralhydrology/evaluation/tester.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_file, m

# Evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluation: 100%|██████████| 1/1 [00:00<00:00,  3.58it/s]
2024-09-18 21:11:02,458: Stored metrics at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/test/model_epoch010/test_metrics.csv
2024-09-18 21:11:02,459: Stored results at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_09484600/test/model_epoch010/test_results.p


,Metric,sota_20,basin_09484600
0,NSE (mean),-0.07,0.01
1,KGE (mean),0.35,0.54
2,MSE (mean),0.00,0.0
3,NSE (median),-0.07,0.01
4,KGE (median),0.35,0.54
5,MSE (median),0.00,0.0


,Metric,sota_20,basin_09484600
0,NSE (mean),-0.07,<b>0.01</b>
1,KGE (mean),0.35,<b>0.54</b>
2,MSE (mean),0.00,<b>0.0</b>
3,NSE (median),-0.07,<b>0.01</b>
4,KGE (median),0.35,<b>0.54</b>
5,MSE (median),0.00,<b>0.0</b>
